# How does the outcome change with dose?

Estimate a response curve, then ask three follow-up questions:

- How steep is the curve at one dose? Use a **local derivative**.
- What percentage change in outcome follows a percentage change in dose? Use **elasticity**.
- What is the average slope across the observed data? Use an **average derivative**.

For each answer, check whether the graph identifies it, whether the data support
it, and what uncertainty is reported.


## Setup

Install Antecedent and the plotting tools from PyPI. In Colab or Jupyter, run:

```python
%pip install antecedent pandas matplotlib
```

These examples use Antecedent 1.10. If you upgrade after importing it, restart
the kernel before continuing.


In [ ]:
# Check that the installed package has the API used in this notebook.
import antecedent

if not all(hasattr(antecedent, name) for name in ("prepare", "load")):
    raise RuntimeError(
        "This notebook requires Antecedent 1.10 or later. Run "
        "%pip install --upgrade antecedent, then restart the kernel."
    )


In [2]:
import numpy as np

import antecedent
from antecedent import (
    AverageDerivative,
    Elasticity,
    PointDerivative,
    ResponseCurve,
    analyze,
)

SEED = 500
N = 1_500

## A confounded nonlinear dose process

Baseline severity affects both the dose selected by clinicians and the outcome. The graph therefore requires adjustment for `severity`. Dose is positive, which gives the elasticity a meaningful treatment scale.

In [3]:
rng = np.random.default_rng(SEED)
severity = rng.normal(size=N)
dose = np.clip(2.5 + 0.65 * severity + rng.normal(scale=0.65, size=N), 0.25, 5.0)
outcome = (
    12.0
    + 3.0 * np.log(dose)
    - 0.30 * dose**2
    + 1.25 * severity
    + rng.normal(scale=0.45, size=N)
)

data = {"severity": severity, "dose": dose, "outcome": outcome}
graph = [
    ("severity", "dose"),
    ("severity", "outcome"),
    ("dose", "outcome"),
]

float(dose.min()), float(np.median(dose)), float(dose.max())

(0.25, 2.5091008806325883, 5.0)

## Ask four questions

Each query takes the treatment name first and the outcome name second.
For derivatives and elasticities, choose a bandwidth in `estimator_config`;
Silverman's rule is not supported for those queries.

The curve includes a dose of `5.25`, beyond the data's maximum of `5.0`.
Watch for the support warning at that point: requesting a dose does not mean
the data can support an estimate there.


In [4]:
curve_query = ResponseCurve(
    "dose",
    "outcome",
    grid=[0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.25],
)
point_query = PointDerivative("dose", "outcome", at=2.5)
elasticity_query = Elasticity("dose", "outcome", at=2.5)
average_query = AverageDerivative("dose", "outcome", weighting="observed")

curve = analyze(data, graph=graph, query=curve_query)
point = analyze(
    data, graph=graph, query=point_query, estimator_config={"bandwidth": 0.35}
)
elasticity = analyze(
    data, graph=graph, query=elasticity_query, estimator_config={"bandwidth": 0.35}
)
average = analyze(data, graph=graph, query=average_query)

### Reuse the analysis and read its report

Keep `result.study` to run the same analysis again. `study.estimate()` uses
its current data; `study.refresh(new_data)` replaces the data after a successful
run. Earlier results stay unchanged.

The report explains the answer and its limitations. Check calibration too:
`unavailable` means no calibration evidence is attached to this result.
A passing diagnostic does not fill that gap.


In [5]:
study = curve.study
report = curve.inspect().to_dict()
print("Answer:", curve.answer)
print("Calibration:", curve.calibration.status, curve.calibration.reason)
report


Answer: Answer(kind='response', value=None, bounds=None, detail=None)
Calibration: unavailable No calibration evidence has been bound to this execution.


{'identification': {'available': True,
  'reason': None,
  'summary': 'nonparametrically_identified',
  'payload': {'status': 'NonparametricallyIdentified',
   'method': 'response.backdoor',
   'adjustment_set': ['severity'],
   'assumption_count': 2,
   'derivation_step_count': 0,
   'horizon_adjustment_sets': None,
   'identified_mass': 1.0,
   'unidentified_mass': 0.0,
   'unevaluable_mass': 0.0,
   'incomplete_search_mass': 0.0,
   'full_mass_scope': True,
   'search_capped': False}},
 'support': {'available': True,
  'reason': None,
  'summary': 'licensed',
  'payload': {'matrix_status': 'licensed',
   'matrix_coordinate': 'ResponseCurve:Dag:explicit:Frequentist:none',
   'empirical': 'unavailable:not_evaluated',
   'execution': {'status': 'outside_empirical_support',
    'query_region': {'dose': [0.5, 5.25]},
    'diagnostics': [{'id': 'response.local_ess',
      'values': [64.784538037472,
       156.7077383011185,
       291.048370223995,
       422.315865667706,
       469.364

## Inspect the orthogonal result axes

An identified estimand can have weak or absent empirical support. Likewise, `pointwise` uncertainty describes coverage at individual grid points; it is not a simultaneous band for the whole curve. The provenance operation identifies the implemented algorithm, while assumptions remain inspectable rather than implicit.

In [6]:
curve.inspect().to_dict()


{'identification': {'available': True,
  'reason': None,
  'summary': 'nonparametrically_identified',
  'payload': {'status': 'NonparametricallyIdentified',
   'method': 'response.backdoor',
   'adjustment_set': ['severity'],
   'assumption_count': 2,
   'derivation_step_count': 0,
   'horizon_adjustment_sets': None,
   'identified_mass': 1.0,
   'unidentified_mass': 0.0,
   'unevaluable_mass': 0.0,
   'incomplete_search_mass': 0.0,
   'full_mass_scope': True,
   'search_capped': False}},
 'support': {'available': True,
  'reason': None,
  'summary': 'licensed',
  'payload': {'matrix_status': 'licensed',
   'matrix_coordinate': 'ResponseCurve:Dag:explicit:Frequentist:none',
   'empirical': 'unavailable:not_evaluated',
   'execution': {'status': 'outside_empirical_support',
    'query_region': {'dose': [0.5, 5.25]},
    'diagnostics': [{'id': 'response.local_ess',
      'values': [64.784538037472,
       156.7077383011185,
       291.048370223995,
       422.315865667706,
       469.364

In [7]:
curve_rows = [
    {"dose": point_values[0], "mean_response": response_values[0]}
    for point_values, response_values in zip(
        curve.response.points, curve.response.values, strict=True
    )
]
curve_rows

[{'dose': 0.5, 'mean_response': 9.622611264271992},
 {'dose': 1.0, 'mean_response': 11.627775189208345},
 {'dose': 1.5, 'mean_response': 12.491029780693456},
 {'dose': 2.0, 'mean_response': 12.82204632174632},
 {'dose': 2.5, 'mean_response': 12.812841170265552},
 {'dose': 3.0, 'mean_response': 12.565926819869949},
 {'dose': 3.5, 'mean_response': 12.017698651982935},
 {'dose': 4.0, 'mean_response': 11.466683357212432},
 {'dose': 4.5, 'mean_response': 10.404972327690704},
 {'dose': 5.25, 'mean_response': -5.752410081281141}]

In [8]:
{
    "local_derivative_at_2_5": point.estimate,
    "elasticity_at_2_5": elasticity.estimate,
    "observed_law_average_derivative": average.estimate,
    "local_support": point.support.status,
    "average_support": average.support.status,
}

{'local_derivative_at_2_5': -0.30428901991610846,
 'elasticity_at_2_5': -0.05928927514847754,
 'observed_law_average_derivative': -0.03184544096804621,
 'local_support': 'supported',
 'average_support': 'supported'}

## Interpretation

The curve describes levels under intervention. The point derivative is the local slope at dose 2.5. The elasticity rescales a local derivative and is not a separate robustness check. The average derivative instead integrates slopes over the observed treatment law, so it answers a population-weighted question.

Do not report the final curve point as empirically supported merely because an estimator returned a number. Inspect `curve.support`, its diagnostics, and its warnings. Also do not describe pointwise intervals as a simultaneous confidence band.